# Development and Microfinance: Evidence of a Middle-Income Peak in Kiva Lending
ECO225 – Project 1  
Kexing Yan  
Professor Khazra
Date: 2026-2-4

## Introduction

Microfinance is often described as a financial tool designed to expand credit access in the poorest economies. However, the relationship between economic development and microfinance activity may not be monotonic. While low-income countries exhibit high credit constraints, extremely weak financial infrastructure and institutional capacity may limit the feasibility of large-scale lending operations. Conversely, in high-income countries, formal financial systems may substitute for microfinance, reducing its relative importance.

This paper examines whether microfinance lending activity follows a nonlinear development pattern, peaking in middle-income countries rather than in the poorest or richest economies. Using country-year data on Kiva lending from 2013 to 2017 merged with World Bank development indicators, we test whether economic development and institutional quality jointly shape the intensity of microfinance activity across countries.

# Project One

## 1.1 Data Cleaning and Data Structure

The dataset combines country-level Kiva lending data with World Bank development indicators. Kiva data were first aggregated to the country-year level for the period 2013–2017. These aggregates were then merged with GDP per capita (constant 2015 USD), population, and institutional quality measures obtained from the World Bank.

The final panel contains 84 countries observed over five years, resulting in 348 country-year observations after merging and removing missing values in core variables.

Key variables include:

- **log_total_loan_amount**: Log of total Kiva loan volume in a country-year.
- **log_gdp_pc**: Log GDP per capita (constant 2015 USD).
- **log_population**: Log population.
- **institutional_pca1**: Composite index of institutional quality derived from governance indicators.

This structure allows us to examine cross-country differences in microfinance activity while controlling for economic scale and institutional environment.

In [2]:
from stargazer.stargazer import Stargazer
from IPython.display import HTML, display

summary_vars = reg[[
    "log_total_loan_amount",
    "log_gdp_pc",
    "institutional_pca1",
    "log_population"
]]

stargazer = Stargazer([summary_vars])
stargazer.title("Summary Statistics")
stargazer.covariate_order([
    "log_total_loan_amount",
    "log_gdp_pc",
    "institutional_pca1",
    "log_population"
])

display(HTML(stargazer.render_html()))

NameError: name 'reg' is not defined

## 1.2 Summary Statistics Tables

Table 1 reports summary statistics for the main variables used in the analysis. The distribution of log GDP per capita spans a wide range, covering low-income to high-income countries. This variation is essential for identifying potential nonlinear patterns in development.

The variation in log total loan amount is substantial across countries and years, indicating meaningful cross-country heterogeneity in microfinance activity. Institutional quality also exhibits considerable dispersion, suggesting that governance conditions differ significantly across the sample.

Overall, the summary statistics confirm that the dataset contains sufficient cross-sectional variation to examine whether microfinance activity follows a nonlinear development pattern.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm

x = reg["log_gdp_pc"]
y = reg["log_total_loan_amount"]

# Quadratic fit
coef = np.polyfit(x, y, 2)
poly = np.poly1d(coef)
x_sorted = np.linspace(x.min(), x.max(), 200)
y_quad = poly(x_sorted)

# Lowess smoother
lowess = sm.nonparametric.lowess(y, x, frac=0.3)

plt.figure()
plt.scatter(x, y, alpha=0.3)
plt.plot(x_sorted, y_quad)
plt.plot(lowess[:,0], lowess[:,1])
plt.xlabel("Log GDP per Capita")
plt.ylabel("Log Total Loan Amount")
plt.title("Microfinance Lending and Economic Development")
plt.show()

## 1.3 Plots and Figures

Figure 1 plots log total loan amount against log GDP per capita. The fitted quadratic curve suggests a nonlinear relationship consistent with an inverted U-shape. Microfinance activity appears to increase with development at low income levels but decline as countries become wealthier.

The LOWESS smoother reinforces this pattern, indicating that lending intensity is not highest in the poorest countries. Instead, activity appears concentrated in lower-middle income economies. This preliminary visual evidence motivates the regression analysis in Project Two.

# Project Two

## 2.1 The Message

Microfinance lending activity follows an inverted U-shaped relationship with economic development. Lending intensity peaks in lower-middle income countries rather than in the poorest or richest economies.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Using quadratic regression coefficients from m5
coef_names = m5.model.exog_names
coef_dict = dict(zip(coef_names, m5.params))

b0 = coef_dict.get("Intercept", 0)
b1 = coef_dict.get("log_gdp_pc", 0)
b2 = coef_dict.get("log_gdp_pc_sq", 0)

x_vals = np.linspace(reg["log_gdp_pc"].min(), reg["log_gdp_pc"].max(), 200)
y_vals = b0 + b1*x_vals + b2*(x_vals**2)

plt.figure()
plt.scatter(reg["log_gdp_pc"], reg["log_total_loan_amount"], alpha=0.25)
plt.plot(x_vals, y_vals)
plt.xlabel("Log GDP per Capita")
plt.ylabel("Predicted Log Loan Amount")
plt.title("Inverted U-Shaped Development Pattern")
plt.show()

Figure 2 presents the fitted quadratic regression relationship between log GDP per capita and log total loan amount. The estimated curve clearly exhibits an inverted U-shape. Microfinance lending increases with development at low income levels, reaches a peak at lower-middle income levels, and declines as countries become wealthier. This pattern is consistent with a development lifecycle interpretation of microfinance activity.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np

# Create per capita lending
df["loan_per_capita"] = df["total_loan_amount"] / df["population"]

# Average across years for mapping
map_df = df.groupby("country_code", as_index=False).mean()

# Load world map
world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))

# Merge
world = world.merge(map_df, left_on="iso_a2", right_on="country_code", how="left")

plt.figure()
world.plot(column="loan_per_capita", legend=True)
plt.title("Microfinance Lending per Capita")
plt.show()

Figure 3 maps average microfinance lending per capita across countries. Lending activity is not uniformly concentrated in the poorest regions. Instead, several lower-middle income countries display relatively high per capita lending intensity, while both extremely low-income and high-income countries exhibit lower levels of activity.

This spatial pattern is consistent with the development lifecycle interpretation presented in Section 2.1.

In [ ]:
plt.figure()
world.plot(column="log_gdp_pc", legend=True)
plt.title("Log GDP per Capita")
plt.show()

Figure 4 maps log GDP per capita across countries. The distribution highlights substantial variation in economic development across the sample, ranging from low-income to high-income economies.

Comparing Figures 3 and 4 suggests that the highest levels of microfinance activity do not occur in the richest economies, reinforcing the nonlinear development hypothesis.

In [ ]:
plt.figure()
world.plot(column="institutional_pca1", legend=True)
plt.title("Institutional Quality (PCA Index)")
plt.show()

Figure 5 maps institutional quality across countries. Governance conditions vary considerably across regions. In several middle-income countries with moderate institutional quality, microfinance activity appears relatively strong.

This pattern is consistent with the regression evidence suggesting that institutional quality moderates the relationship between development and lending activity.

In [ ]:
f1 = "log_total_loan_amount ~ log_gdp_pc"
f2 = "log_total_loan_amount ~ log_gdp_pc + institutional_pca1"
f3 = "log_total_loan_amount ~ log_gdp_pc + institutional_pca1 + log_population"
f4 = "log_total_loan_amount ~ log_gdp_pc * institutional_pca1 + log_population"

m1 = fit_ols(f1, reg)
m2 = fit_ols(f2, reg)
m3 = fit_ols(f3, reg, se="cluster", cluster_col="country_code")
m4 = fit_ols(f4, reg, se="cluster", cluster_col="country_code")

In [ ]:
from stargazer.stargazer import Stargazer
from IPython.display import HTML, display

stargazer1 = Stargazer([m1, m2, m3, m4])
stargazer1.title("Baseline Regressions and Institutional Interaction")
display(HTML(stargazer1.render_html()))

Table 2 presents baseline regression results. In the linear specification, the coefficient on log GDP per capita is not consistently significant, suggesting that a simple monotonic relationship may not adequately describe microfinance activity.

Including institutional quality and population controls does not materially change this result. However, the interaction specification indicates that institutional quality moderates the development effect. In countries with stronger institutions, increases in income are associated with a more pronounced decline in microfinance activity, consistent with substitution by formal financial systems.

These findings motivate a nonlinear specification to capture potential development lifecycle effects.

In [ ]:
# Quadratic
reg["log_gdp_pc_sq"] = reg["log_gdp_pc"]**2

f5 = "log_total_loan_amount ~ log_gdp_pc + log_gdp_pc_sq + institutional_pca1 + log_population"
m5 = fit_ols(f5, reg, se="cluster", cluster_col="country_code")

# Year FE
f6 = "log_total_loan_amount ~ log_gdp_pc + institutional_pca1 + log_population + C(year)"
m6 = fit_ols(f6, reg, se="cluster", cluster_col="country_code")

# Alternative institutional index
reg2 = df[[
    "log_total_loan_amount",
    "log_gdp_pc",
    "institutional_index",
    "log_population",
    "year",
    "country_code"
]].dropna().copy()

f7 = "log_total_loan_amount ~ log_gdp_pc + institutional_index + log_population"
m7 = fit_ols(f7, reg2, se="cluster", cluster_col="country_code")

# Financial access
reg3 = df[[
    "log_total_loan_amount",
    "log_gdp_pc",
    "financial_access_index",
    "log_population",
    "year",
    "country_code"
]].dropna().copy()

f8 = "log_total_loan_amount ~ log_gdp_pc + financial_access_index + log_population"
m8 = fit_ols(f8, reg3, se="cluster", cluster_col="country_code")

In [ ]:
stargazer2 = Stargazer([m5, m6, m7, m8])
stargazer2.title("Nonlinearity and Robustness Specifications")
display(HTML(stargazer2.render_html()))

Table 3 introduces a quadratic specification in log GDP per capita. The positive coefficient on log GDP per capita and the negative coefficient on its squared term indicate an inverted U-shaped relationship between development and microfinance activity.

The implied turning point occurs at approximately $1,700 GDP per capita, corresponding to lower-middle income economies. This finding supports the development lifecycle interpretation: microfinance activity rises with development at low income levels but declines as countries become wealthier.

Including year fixed effects does not eliminate this nonlinear pattern, although overall variation is partly driven by global expansion of the Kiva platform over time.

Alternative institutional measures and financial access indicators yield qualitatively similar conclusions, suggesting that the nonlinear development pattern is robust across specifications.

## 2.4 Conclusion

This project examines how economic development relates to microfinance lending activity using country-year Kiva data (2013–2017) merged with World Bank indicators. The central finding is that microfinance lending follows a **nonlinear development pattern**: activity is highest in **lower-middle income** economies rather than in the poorest or richest countries. The quadratic specification provides evidence of an inverted U-shape, with the implied peak occurring around **$1,700 GDP per capita**, consistent with a development “lifecycle” interpretation of microfinance.

A plausible mechanism is that very low-income countries face operational constraints—weak infrastructure, higher risk, and limited absorptive capacity—that restrict lending scale even when credit needs are large. As countries reach middle-income status, basic institutional and market conditions may become sufficient for microfinance platforms to operate at higher volume. At higher income levels, microfinance activity declines as formal financial systems expand and substitute for microfinance. The interaction results support this interpretation: stronger institutional quality is associated with a steeper decline in microfinance activity as income rises.

Several limitations are important. First, the panel is short (2013–2017), and year fixed effects indicate that global platform expansion contributes meaningfully to variation in lending. Second, this analysis is descriptive and does not claim a fully causal effect of development on lending. Third, robustness checks involving poverty rates rely on a smaller sample due to missingness, which reduces precision. Despite these constraints, the evidence consistently supports the main message that microfinance is most active in middle-income economies rather than being concentrated exclusively in the poorest countries.